# 07 模型网关、观测与成本治理

模型网关和观测系统的价值是把“到处散落的模型调用”变成“可控、可追踪、可治理的平台能力”。没有这一层，模型越多、团队越多、供应商越多，成本和风险越不可控。


## 1. 模型网关解决什么

LiteLLM 这类网关的核心能力包括：统一 OpenAI 格式、provider 路由、retry/fallback、virtual keys、预算、rate limit、logging/observability callbacks。自研网关也通常要做类似能力。

```mermaid
flowchart LR
    App1["应用 A / App A"] --> GW["模型网关 / LLM Gateway"]
    App2["应用 B / App B"] --> GW
    GW --> Auth["虚拟密钥与鉴权 / virtual keys / auth"]
    GW --> Budget["预算与配额 / budget / quota"]
    GW --> Route["路由与降级 / routing / fallback"]
    Route --> Local["本地 vLLM/SGLang / local serving"]
    Route --> Cloud["云模型供应商 / cloud providers"]
    GW --> Obs["日志/指标/追踪 / logs / metrics / traces"]
```

网关不是为了增加复杂度，而是为了把鉴权、限流、成本、fallback、审计从业务代码里抽出来。


## 2. 路由、fallback 与预算

常见策略：

- **按任务路由**：简单分类走小模型，复杂推理走大模型。
- **按租户路由**：不同部门、客户、环境有不同模型和预算。
- **按健康状态 fallback**：本地模型超时或错误时切到云模型。
- **按成本路由**：先用便宜模型，不确定时升级。
- **按数据安全路由**：敏感数据只能走私有部署。

风险：fallback 可能让成本突然上升；多模型切换可能让输出风格和质量不一致；预算限制如果没有良好错误提示，会影响用户体验。


In [ ]:
# 一个最小路由器：根据任务、敏感性、预算选择模型。
models = {
    'local_small': {'cost': 0.1, 'private': True, 'quality': 0.65},
    'local_large': {'cost': 0.6, 'private': True, 'quality': 0.82},
    'cloud_best': {'cost': 2.0, 'private': False, 'quality': 0.92},
}

def route(task_complexity, sensitive, remaining_budget):
    candidates = []
    for name, m in models.items():
        if sensitive and not m['private']:
            continue
        if m['cost'] <= remaining_budget:
            candidates.append((name, m))
    if not candidates:
        return 'reject_or_degrade'
    target_quality = 0.8 if task_complexity == 'hard' else 0.6
    good = [x for x in candidates if x[1]['quality'] >= target_quality]
    return min(good or candidates, key=lambda x: x[1]['cost'])[0]

for case in [('easy', False, 0.5), ('hard', False, 3.0), ('hard', True, 1.0), ('hard', True, 0.2)]:
    print(case, '->', route(*case))


## 3. Observability：metrics、logs、traces

- **Metrics**：聚合指标，适合 dashboard 和告警。例如 TTFT、ITL、QPS、error rate、token rate、cost。
- **Logs**：离散事件，适合查某个 request id 的细节。例如模型、状态码、token 数、错误信息。
- **Traces**：一次请求的分段链路，适合定位慢在哪里。例如 retrieval span、rerank span、model span、tool span。

OpenTelemetry GenAI 语义约定正在定义 GenAI 的 spans、metrics、events、exceptions，包括 model operations、agent/framework spans、input/output events 等。它的意义是让不同框架、网关和观测平台使用统一字段，降低排障成本。


## 4. GenAI trace 应该长什么样

```mermaid
sequenceDiagram
    participant App
    participant Retriever
    participant Gateway
    participant Model
    participant Tool
    App->>Retriever: retrieval span
    Retriever-->>App: docs + scores
    App->>Gateway: model span starts
    Gateway->>Model: chat completion
    Model-->>Gateway: streamed tokens
    Gateway-->>App: usage + latency
    App->>Tool: tool span (optional)
    Tool-->>App: result / error
```

一个好的 trace 至少要能回答：用户问了什么、检索到了什么、prompt 版本是什么、调用了哪个模型、token 数是多少、每段耗时多少、是否触发 fallback、工具调用是否成功。


In [ ]:
# 用 mock trace 定位慢在哪里。
trace = [
    {'span': 'auth', 'ms': 6, 'status': 'ok'},
    {'span': 'retrieval', 'ms': 120, 'status': 'ok'},
    {'span': 'rerank', 'ms': 80, 'status': 'ok'},
    {'span': 'model_queue', 'ms': 450, 'status': 'ok'},
    {'span': 'model_prefill', 'ms': 900, 'status': 'ok'},
    {'span': 'model_decode', 'ms': 1800, 'status': 'ok'},
]

total = sum(s['ms'] for s in trace)
for s in sorted(trace, key=lambda x: x['ms'], reverse=True):
    print(f"{s['span']:14s} {s['ms']:4d}ms {s['ms']/total:5.1%}")


## 5. 事故排查 walkthrough

场景：用户反馈“今天知识库助手特别慢”。排查步骤：

1. 看 dashboard：TTFT 变高还是 ITL 变高？错误率是否上升？token rate 是否异常？
2. 按租户/模型/版本切分：是否某个部门、模型或发布版本导致？
3. 查 trace：慢在 retrieval、rerank、model queue、prefill 还是 decode？
4. 查网关：是否 fallback 到云模型？是否预算/限流触发？
5. 查 serving：KV cache 是否接近满？是否出现 preemption？GPU 是否 OOM 或重启？
6. 缓解：限流、降级、缩短上下文、关闭新版本、扩容、切 fallback。
7. 复盘：补指标、补评测、补告警、补灰度策略。

```mermaid
flowchart TB
    Alert["延迟告警 / latency alert"] --> Slice["按租户/模型/版本切分"]
    Slice --> Trace["查看 trace 分段"]
    Trace --> Cause["定位瓶颈"]
    Cause --> Mitigate["缓解: rollback/scale/limit/fallback"]
    Mitigate --> Review["复盘: 指标/评测/门禁"]
```


## 6. 常见误区

- **只记录 prompt 和 output，不记录版本。** 没有模型/prompt/data 版本，质量问题无法复现。
- **把敏感内容直接打进日志。** prompt logging 要有脱敏、采样、权限和保留策略。
- **fallback 不计成本。** 云模型 fallback 可能在故障时放大成本。
- **只看模型指标，不看检索和工具。** Agent/RAG 的慢和错经常发生在模型外。


## 7. 成本治理：token 是账单单位也是系统压力单位

LLM 成本通常和输入/输出 token 相关。即使本地模型没有按 token 付费，token 也代表 GPU 时间、显存和机会成本。成本治理常见手段：

- 设置 `max_tokens` 和上下文上限。
- 对长文档做摘要/压缩，而不是全塞进 prompt。
- 简单任务走小模型，复杂任务升级。
- 对不同租户设置 budget 和 rate limit。
- 缓存高频请求或共享 prefix。
- fallback 必须计入预算，避免故障时成本爆炸。

成本不是财务问题，成本是架构问题。


In [ ]:
# 简单 token 成本估算。单位价格只是示例，不代表任何供应商。
prices = {
    'small': {'input_per_1k': 0.0001, 'output_per_1k': 0.0002},
    'large': {'input_per_1k': 0.0010, 'output_per_1k': 0.0030},
}
requests = [
    {'model': 'small', 'input': 800, 'output': 200},
    {'model': 'large', 'input': 5000, 'output': 500},
    {'model': 'large', 'input': 9000, 'output': 1000},
]
for r in requests:
    p = prices[r['model']]
    cost = r['input']/1000*p['input_per_1k'] + r['output']/1000*p['output_per_1k']
    print(r, 'cost=', round(cost, 5))


## 8. Prompt logging 的安全边界

观测需要记录足够信息来排查问题，但 prompt 和输出可能包含敏感数据。实践上可以分层：

- 默认记录 metadata：request id、tenant、model、token、latency、status。
- 对内容做采样，不是 100% 全量记录。
- 对 PII、密钥、身份证、手机号等做脱敏。
- 对敏感租户关闭内容日志，只保留 hash 或外部安全存储引用。
- 设置日志保留周期和访问权限。

这也是为什么 OpenTelemetry GenAI 语义约定会区分 spans、events、attributes，并提示内容体积和敏感性问题。观测不是越多越好，而是足够定位且风险可控。


## 9. 多租户治理

企业内部 AI 平台通常服务多个团队。没有多租户治理时，常见问题是：某个团队高并发拖慢所有人，某个实验用光预算，敏感部门日志被无关人员看到，测试环境误调用生产模型。

多租户治理至少包括：virtual key、tenant id、rate limit、budget、模型白名单、日志权限、数据隔离、优先级和审计。网关是实现这些能力的自然位置，因为所有模型调用都经过它。


## 10. 可观测字段设计

建议每次模型调用至少记录以下 metadata：

| 字段 | 用途 |
|---|---|
| request_id | 串联业务、网关、模型、用户反馈 |
| tenant_id / user_id hash | 多租户切分和限流 |
| model / provider / route | 判断是否 fallback 或模型切换 |
| prompt_version / app_version | 复现质量问题 |
| input_tokens / output_tokens | 成本和容量分析 |
| ttft / total_latency / status | 性能与可靠性 |
| retrieval_doc_ids | RAG 证据追踪 |
| tool_names / tool_status | Agent 工具排障 |

内容日志要谨慎，但 metadata 应尽量完整。


## 11. 自测题

1. 为什么 fallback 既是可靠性手段，也是成本风险？
2. metrics、logs、traces 分别解决什么问题？
3. prompt logging 为什么要采样和脱敏？
4. OpenTelemetry GenAI 语义约定的价值是什么？
5. 如果用户投诉某次回答错误，你需要哪些字段才能复盘？


## 12. 质量、性能、成本要一起看

AI 系统的难点在于质量、性能和成本经常互相拉扯。换大模型可能提升质量，但增加延迟和成本；缩短上下文能降低 TTFT 和成本，但可能降低答案完整性；fallback 能提升可用性，但可能导致账单上升；记录完整 prompt 有助于排障，但增加隐私风险。

所以 dashboard 不能只有技术指标，也要有产品指标。建议至少同时看：成功率、用户满意度或人工纠错率、TTFT/ITL、token cost、fallback rate、检索命中率、工具失败率。只有把这些指标放在一起，团队才能做理性取舍。

例如成本上升不一定是坏事，如果同时质量显著提升且业务转化更好，可能值得；TTFT 下降也不一定是好事，如果因为过度截断上下文导致答案错误率上升，就只是把问题从性能转移到质量。AI Infra 的治理目标不是某个单指标最优，而是系统整体可控。


## 来源与延伸阅读

本章正文已经把关键内容消化进教程，下面链接只用于延伸阅读和核对最新细节：vLLM、vLLM-Metal、Ray Serve LLM、SGLang、TensorRT-LLM、Text Generation Inference、LiteLLM、KServe、NVIDIA GPU Operator、OpenTelemetry GenAI。
